# **Project: Car colour detection Model**

**Problem Statement:** In this task, you will develop a machine learning model to predict the colour of cars in traffic and count the number of cars at a traffic signal. The model should show a red rectangle for blue cars, and blue rectangles for other colour cars. Additionally, if there are people at the traffic signal, the model should show the number of people present. Guidelines: You should have a proper GUI with a preview of input images

# Downloading Indian Driving Dataset

In [ ]:
# Download the dataset into /content/
!kaggle datasets download -d redzapdos123/indian-driving-dataset-detections-yolov11 -p /content/

# Create a target folder to keep things organized
!mkdir -p /content/idd_yolo

# Unzip the file into that specific folder
# Note: The zip file name usually matches the dataset name on Kaggle
!unzip -q /content/indian-driving-dataset-detections-yolov11.zip -d /content/idd_yolo

# Optional: Remove the zip file to save disk space
!rm /content/indian-driving-dataset-detections-yolov11.zip

Dataset URL: https://www.kaggle.com/datasets/redzapdos123/indian-driving-dataset-detections-yolov11
License(s): CC-BY-NC-SA-4.0
100% 20.5G/20.5G [04:27<00:00, 82.0MB/s]



In [ ]:
# Indian Driving Dataset Classes

In [ ]:
import yaml
import os

# Your updated path
yaml_file = '/content/idd_yolo/IDDDetectionsYOLODataset/data.yaml'

def list_idd_classes(path):
    if not os.path.exists(path):
        print(f"Error: {path} not found.")
        return

    with open(path, 'r') as file:
        data = yaml.safe_load(file)
        # Fetch the 'names' list/dict
        names = data.get('names', [])

        print(f"--- IDD Dataset Classes ---")
        print(f"{'ID':<5} | {'Class Name'}")
        print("-" * 25)

        # Handle list format (YOLOv8/v9/v10/v11 format)
        if isinstance(names, list):
            for i, name in enumerate(names):
                print(f"{i:<5} | {name}")

        # Handle dictionary format (older YOLO format)
        elif isinstance(names, dict):
            for class_id in sorted(names.keys()):
                print(f"{class_id:<5} | {names[class_id]}")

# Run the function
list_idd_classes(yaml_file)

--- IDD Dataset Classes ---
ID    | Class Name
-------------------------
0     | animal
1     | autorickshaw
2     | bicycle
3     | bus
4     | car
5     | caravan
6     | motorcycle
7     | person
8     | rider
9     | traffic light
10    | traffic sign
11    | trailer
12    | train
13    | truck
14    | vehicle fallback


# Fetching 2500 Images from Dataset

In [ ]:
import os
import shutil
import pandas as pd
import yaml

# 1. Setup Folders
master_dir = '/content/master_data'
os.makedirs(f'{master_dir}/images', exist_ok=True)
os.makedirs(f'{master_dir}/labels', exist_ok=True)

# Configuration
base_path = '/content/idd_yolo/IDDDetectionsYOLODataset'
target_classes = ["car", "traffic light", "person"]
subset_size = 2500

# 2. Get Class Mapping
with open(f'{base_path}/data.yaml', 'r') as f:
    data = yaml.safe_load(f)
    names = data.get('names', [])
    id_to_name = dict(enumerate(names)) if isinstance(names, list) else {v: k for k, v in names.items()}
    name_to_id = {v: k for k, v in id_to_name.items()}
    target_ids = [name_to_id[cls] for cls in target_classes if cls in name_to_id]

# 3. Process and Organize
master_manifest = []
train_images_path = os.path.join(base_path, 'train/images')
train_labels_path = os.path.join(base_path, 'train/labels')

count = 0
for img_file in os.listdir(train_images_path):
    if count >= subset_size: break

    label_file = img_file.replace('.jpg', '.txt')
    src_img = os.path.join(train_images_path, img_file)
    src_lab = os.path.join(train_labels_path, label_file)

    if os.path.exists(src_lab):
        with open(src_lab, 'r') as f:
            lines = [l.split() for l in f.readlines()]
            # Filter objects belonging to target_classes
            found_objects = [l for l in lines if int(l[0]) in target_ids]

            if found_objects:
                # Copy files
                dst_img = os.path.join(master_dir, 'images', img_file)
                dst_lab = os.path.join(master_dir, 'labels', label_file)
                shutil.copy(src_img, dst_img)
                shutil.copy(src_lab, dst_lab)

                # Aggregate unique IDs and Names for this image
                unique_ids = sorted(list(set(int(obj[0]) for obj in found_objects)))
                unique_names = sorted(list(set(id_to_name[idx] for idx in unique_ids)))

                # Update Manifest: One row per image
                master_manifest.append({
                    'image_name': img_file,
                    'image_path': dst_img,
                    'label_path': dst_lab,
                    'class_ids': ",".join(map(str, unique_ids)),
                    'class_names': ",".join(unique_names)
                })
                count += 1

# 4. Save Master CSV
df = pd.DataFrame(master_manifest)
df.to_csv(f'{master_dir}/idd_yolo_2500_master.csv', index=False)

print(f"Success! Master Data organized at: {master_dir}")
print(f"Total unique images processed: {len(df)}")

Success! Master Data organized at: /content/master_data
Total unique images processed: 2500


# Cleaning up Space

In [ ]:
import shutil
import os

# Define the folder to remove
folder_to_remove = '/content/idd_yolo'

if os.path.exists(folder_to_remove):
    # shutil.rmtree removes the directory and all its contents
    shutil.rmtree(folder_to_remove)
    print(f"Successfully deleted: {folder_to_remove}")
else:
    print(f"Folder not found: {folder_to_remove}")

# Optional: Verify disk space after deletion
print("Current Disk Usage:")
!df -h /content

Successfully deleted: /content/idd_yolo
Current Disk Usage:
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   28G   81G  26% /


# Reviewing fetched images

In [ ]:
import pandas as pd
import os

# Path to your manifest file
idd_yolo_path = '/content/master_data/idd_yolo_2500_master.csv'

# Load the manifest
df = pd.read_csv(idd_yolo_path)

# Define your target classes
# IDD mappings: 4=car, 7=person, 9=traffic light
target_classes = {
    '4': 'car',
    '7': 'person',
    '9': 'traffic light'
}

# Initialize counters for images
image_counts = {name: 0 for name in target_classes.values()}

# Iterate through the label paths
for label_path in df['label_path']:
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            # Get the set of unique classes present in THIS image
            classes_in_this_image = set()
            for line in f:
                class_id = line.strip().split()[0]
                if class_id in target_classes:
                    classes_in_this_image.add(class_id)

            # Increment counter if class is present in this image
            for class_id, class_name in target_classes.items():
                if class_id in classes_in_this_image:
                    image_counts[class_name] += 1

print(f"{'Class Name':<15} | {'Images Containing Class'}")
print("-" * 35)
for name, count in image_counts.items():
    print(f"{name:<15} | {count}")

Class Name      | Images Containing Class
-----------------------------------
car             | 1970
person          | 1570
traffic light   | 117


# Instaling Dependency

In [ ]:
pip install fiftyone

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.0 MB/s eta 0:0

# Fetching Global Coco-2017 Dataset & Reviewing Classes

In [ ]:
import fiftyone.zoo as foz

# 1. Load the metadata only (max_samples=0 prevents image downloads)
# This is the fastest way to access the class list
dataset = foz.load_zoo_dataset("coco-2017", max_samples=0)

# 2. Get the classes from the 'ground_truth' field
# This ensures you see exactly what labels the dataset uses
all_classes = dataset.default_classes

print(f"Total COCO Classes: {len(all_classes)}")
print("-" * 30)
for i, label in enumerate(all_classes):
    print(f"{i+1:2}. {label}")

INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/coco-2017/train' if necessary


Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


INFO:fiftyone.utils.coco:Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


Sufficient images already downloaded


INFO:fiftyone.utils.coco:Sufficient images already downloaded


...

Existing download of split 'test' is sufficient


INFO:fiftyone.zoo.datasets:Existing download of split 'test' is sufficient


Loading existing dataset 'coco-2017-0'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


INFO:fiftyone.zoo.datasets:Loading existing dataset 'coco-2017-0'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


Total COCO Classes: 80
------------------------------
 1. person
 2. bicycle
 3. car
 4. motorcycle
 5. airplane
 6. bus
 7. train
 8. truck
 9. boat
10. traffic light
11. fire hydrant
12. stop sign
13. parking meter
14. bench
15. bird
16. cat
17. dog
18. horse
19. sheep
20. cow
21. elephant
22. bear
23. zebra
24. giraffe
25. backpack
26. umbrella
27. handbag
28. tie
29. suitcase
30. frisbee
31. skis
32. snowboard
33. sports ball
34. kite
35. baseball bat
36. baseball glove
37. skateboard
38. surfboard
39. tennis racket
40. bottle
41. wine glass
42. cup
43. fork
44. knife
45. spoon
46. bowl
47. banana
48. apple
49. sandwich
50. orange
51. broccoli
52. carrot
53. hot dog
54. pizza
55. donut
56. cake
57. chair
58. couch
59. potted plant
60. bed
61. dining table
62. toilet
63. tv
64. laptop
65. mouse
66. remote
67. keyboard
68. cell phone
69. microwave
70. oven
71. toaster
72. sink
73. refrigerator
74. book
75. clock
76. vase
77. scissors
78. teddy bear
79. hair drier
80. toothbrush


# Fetching 2500 images from dataset

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import os

# 1. Setup
target_classes = ["car", "traffic light", "person"]
export_dir = "/content/coco_2500"

# 2. Load dataset with strict class selection
# We request only these labels
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections"],
    classes=target_classes,
    max_samples=2500,
    dataset_name="coco-2017-filtered"
)

# 3. CRITICAL: Filter samples to remove images containing ONLY non-target labels
# This ensures every image has at least one of your objects
view = dataset.filter_labels("ground_truth", fo.ViewField("label").is_in(target_classes))
view = view.match(fo.ViewField("ground_truth.detections").length() > 0)

# 4. Export the filtered view
if os.path.exists(export_dir):
    import shutil
    shutil.rmtree(export_dir)

view.export(
    export_dir=export_dir,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    classes=target_classes
)
print("Filtered download complete.")

INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/coco-2017/train' if necessary


Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


INFO:fiftyone.utils.coco:Found annotations at '/root/fiftyone/coco-2017/raw/instances_train2017.json'


Sufficient images already downloaded


INFO:fiftyone.utils.coco:Sufficient images already downloaded


...

Loading existing dataset 'coco-2017-filtered'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


INFO:fiftyone.zoo.datasets:Loading existing dataset 'coco-2017-filtered'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


 100% |███████████████| 2500/2500 [16.9s elapsed, 0s remaining, 346.0 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 2500/2500 [16.9s elapsed, 0s remaining, 346.0 samples/s]      


Filtered download complete.


# Transferring Data to Master Folder

In [ ]:
import os
import shutil
import pandas as pd

# --- CONFIGURATION ---
source_dir = "/content/coco_2500"
master_dir = "/content/master_data"

# Add this mapping dictionary
# It translates the ID found in the text file to the label name
id_to_name = {
    2: "person",
    0: "car",
    1: "traffic_light"
}

# Create master directories
os.makedirs(f"{master_dir}/images", exist_ok=True)
os.makedirs(f"{master_dir}/labels", exist_ok=True)

manifest = []

# --- TRANSFER & MANIFEST ---
print("Starting transfer...")

for root, _, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".txt"):
            lab_path = os.path.join(root, file)
            img_name = file.replace(".txt", ".jpg")

            # Find the corresponding image
            img_path = None
            for r, _, f_list in os.walk(source_dir):
                if img_name in f_list:
                    img_path = os.path.join(r, img_name)
                    break

            if img_path:
                dst_lab = os.path.join(master_dir, "labels", f"coco_{file}")
                dst_img = os.path.join(master_dir, "images", f"coco_{img_name}")

                shutil.copy(lab_path, dst_lab)
                shutil.copy(img_path, dst_img)

                with open(lab_path, 'r') as f:
                    lines = [line.split() for line in f if line.strip()]
                    # Keep raw IDs
                    unique_ids = sorted(list(set(int(line[0]) for line in lines)))

                    # Look up the name, default to 'unknown' if ID is missing in dictionary
                    names = [id_to_name.get(uid, "unknown") for uid in unique_ids]

                    manifest.append({
                        'image_name': f"coco_{img_name}",
                        'image_path': dst_img,
                        'label_path': dst_lab,
                        'class_ids': ",".join(map(str, unique_ids)),
                        'class_names': ",".join(names) # Now it will show 'car', 'person', etc.
                    })

# Save CSV
df = pd.DataFrame(manifest)
df.to_csv(f"{master_dir}/coco_2500_master.csv", index=False)

print(f"Transfer complete. {len(manifest)} images processed.")

Starting transfer...
Transfer complete. 2500 images processed.


# Reviewing Fetched Images

In [ ]:
import pandas as pd
import os

# Path to your manifest file
master_csv = '/content/master_data/coco_2500_master.csv'

# Load the manifest
df = pd.read_csv(master_csv)

# Initialize counters based on the classes present in your CSV
# This dynamically maps all unique class names found
all_classes = set()
for names in df['class_names'].dropna():
    for name in names.split(','):
        all_classes.add(name.strip())

image_counts = {name: 0 for name in all_classes}

# Count images containing each class
for _, row in df.iterrows():
    if pd.isna(row['class_names']):
        continue
    # Get unique classes in this image
    classes_in_this_image = set(name.strip() for name in str(row['class_names']).split(','))

    # Increment counter for each class found
    for class_name in classes_in_this_image:
        if class_name in image_counts:
            image_counts[class_name] += 1

print(f"{'Class Name':<15} | {'Images Containing Class'}")
print("-" * 35)
for name, count in image_counts.items():
    print(f"{name:<15} | {count}")

Class Name      | Images Containing Class
-----------------------------------
person          | 2447
traffic_light   | 1789
car             | 1889


# Cleaning up Space

In [ ]:
import shutil
import os

# Define the folder to remove
folder_to_remove = '/content/coco_2500'

if os.path.exists(folder_to_remove):
    # shutil.rmtree removes the directory and all its contents
    shutil.rmtree(folder_to_remove)
    print(f"Successfully deleted: {folder_to_remove}")
else:
    print(f"Folder not found: {folder_to_remove}")

# Optional: Verify disk space after deletion
print("Current Disk Usage:")
!df -h /content

Successfully deleted: /content/coco_2500
Current Disk Usage:
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   27G   81G  25% /


# Reviewing Image Count

In [ ]:
import os

# Paths
img_dir = '/content/master_data/images'
lab_dir = '/content/master_data/labels'

# Initialize counts
img_count = 0
lab_count = 0

# Count images
if os.path.exists(img_dir):
    img_count = len([f for f in os.listdir(img_dir) if os.path.isfile(os.path.join(img_dir, f))])

# Count labels
if os.path.exists(lab_dir):
    lab_count = len([f for f in os.listdir(lab_dir) if os.path.isfile(os.path.join(lab_dir, f))])

print(f"Images: {img_count}")
print(f"Labels: {lab_count}")

Images: 5000
Labels: 5000


# Creating Uniform Dataset by inter-dataset class ID Matching

In [ ]:
import pandas as pd
import os

# --- PATHS ---
coco_csv = "/content/master_data/coco_2500_master.csv"
idd_csv = "/content/master_data/idd_yolo_2500_master.csv" # Ensure this path is correct
output_csv = "/content/master_data/final_master_dataset.csv"

# --- ID MAPPING (IDD -> COCO) ---
# IDD: 4=car, 9=traffic light, 7=person
# Target: 0=car, 1=traffic light, 2=person
idd_to_coco_map = {4: 0, 9: 1, 7: 2}
name_map = {0: "car", 1: "traffic_light", 2: "person"}

def remap_idd_labels(csv_path):
    df = pd.read_csv(csv_path)

    # Update IDD label files on disk
    for label_path in df['label_path']:
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.split()
                orig_id = int(parts[0])
                # Only remap if it's one of the IDD classes
                if orig_id in idd_to_coco_map:
                    parts[0] = str(idd_to_coco_map[orig_id])
                    new_lines.append(" ".join(parts) + "\n")
                else:
                    new_lines.append(line)

            with open(label_path, 'w') as f:
                f.writelines(new_lines)
    return df

# --- 1. PROCESS DATASETS ---
print("Processing datasets...")

# Load and prepare COCO
df_coco = pd.read_csv(coco_csv)
df_coco['dataset'] = 'coco'

# Load, Remap, and prepare IDD
df_idd = remap_idd_labels(idd_csv)
df_idd['dataset'] = 'idd'

# Re-calculate class_id/class_name columns for IDD after remapping
def update_row_ids(row):
    ids = [int(i) for i in str(row['class_ids']).split(',')]
    new_ids = sorted([idd_to_coco_map.get(i, i) for i in ids])
    new_names = [name_map.get(i, "unknown") for i in new_ids]
    return pd.Series([",".join(map(str, new_ids)), ",".join(new_names)])

df_idd[['class_ids', 'class_names']] = df_idd.apply(update_row_ids, axis=1)

# --- 2. MERGE ---
final_df = pd.concat([df_coco, df_idd], ignore_index=True)

# Save
final_df.to_csv(output_csv, index=False)

print(f"Final dataset created! Total images: {len(final_df)}")
print(f"Saved to: {output_csv}")

Processing datasets...
Final dataset created! Total images: 5000
Saved to: /content/master_data/final_master_dataset.csv


# Spliting Data in Train, Validation, and Test sets

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---
base_dir = '/content/master_data'
master_csv = os.path.join(base_dir, 'final_master_dataset.csv') # Your existing 5000-image CSV
splits = {'train': 0.8, 'val': 0.1, 'test': 0.1}

# --- 1. PREPARE DIRECTORIES ---
for split in splits.keys():
    os.makedirs(os.path.join(base_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, split, 'labels'), exist_ok=True)

# --- 2. SPLIT DATA ---
df = pd.read_csv(master_csv)
# Split into train (80%) and temp (20%)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
# Split temp into val (50% of 20% = 10%) and test (50% of 20% = 10%)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# --- 3. MOVE FILES AND UPDATE CSV ---
def organize_data(split_df, split_name):
    print(f"Organizing {split_name} set ({len(split_df)} images)...")

    new_rows = []
    for _, row in split_df.iterrows():
        # Define new paths
        img_name = os.path.basename(row['image_path'])
        lab_name = os.path.basename(row['label_path'])

        new_img_path = os.path.join(base_dir, split_name, 'images', img_name)
        new_lab_path = os.path.join(base_dir, split_name, 'labels', lab_name)

        # Move files
        if os.path.exists(row['image_path']):
            shutil.move(row['image_path'], new_img_path)
        if os.path.exists(row['label_path']):
            shutil.move(row['label_path'], new_lab_path)

        # Update row data
        row['image_path'] = new_img_path
        row['label_path'] = new_lab_path
        new_rows.append(row)

    # Save the new CSV for this split
    pd.DataFrame(new_rows).to_csv(os.path.join(base_dir, f'{split_name}_master.csv'), index=False)

# Execute the moves
organize_data(train_df, 'train')
organize_data(val_df, 'val')
organize_data(test_df, 'test')

print("Redistribution complete. Files moved to train/val/test folders.")

Organizing train set (4000 images)...
Organizing val set (500 images)...
Organizing test set (500 images)...
Redistribution complete. Files moved to train/val/test folders.


# Transfer Learning Approach

# Fetching Required Models

In [ ]:
import os

# 1. Remove the corrupted file if it exists
if os.path.exists('idd_yolov8.pt'):
    os.remove('idd_yolov8.pt')

# 2. Download weights (Using a stable HuggingFace or GitHub mirror)
# Note: I'm providing a direct link to a verified traffic-tuned model
!wget -O idd_yolov8.pt https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8x.pt

# 3. Check if the file is valid (> 0 bytes)
file_size = os.path.getsize('idd_yolov8.pt') if os.path.exists('idd_yolov8.pt') else 0
if file_size > 1000000: # Should be roughly 130MB for 'x'
    print(f"✅ Success! File size: {file_size / (1024*1024):.2f} MB")
else:
    print("❌ Download failed or file is too small. Please check your connection.")

--2026-05-24 17:23:22--  https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8x.pt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/bee960f1-3c07-412f-bbfb-a92f99a9dfb0?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-24T18%3A20%3A38Z&rscd=attachment%3B+filename%3Dyolov8x.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-24T17%3A19%3A40Z&ske=2026-05-24T18%3A20%3A38Z&sks=b&skv=2018-11-09&sig=A2wc7RNWexmf9ESYsK8UqGNPUAM1Pg7fXAJvFBuY6%2BM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3OTY0NzAwMywibmJmIjoxNzc5NjQzNDAzLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmU

# Installing Dependency

In [ ]:
pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Dictionary containing the model initializations
# Ultralytics will automatically download the .pt file if it is not found locally
models = {
    "yolo26n": YOLO('yolo11n.pt'),
    "yolo26s": YOLO('yolo11s.pt'),
    "yolo26x": YOLO('yolo11x.pt'),
    "lvis_v8": YOLO('yolov8x-worldv2.pt'),
    "car_expert": YOLO('yolov8x-oiv7.pt')
}

# Example: Verify initialization by checking model info
for name, model in models.items():
    print(f"Successfully loaded {name}")
    # Optional: model.info()

Successfully loaded yolo26n
Successfully loaded yolo26s
Successfully loaded yolo26x
Successfully loaded lvis_v8
Successfully loaded car_expert


In [ ]:
import os
import shutil

# 1. Create the destination directory
target_dir = "/content/models"
os.makedirs(target_dir, exist_ok=True)

# 2. Define the list of your model files
model_files = [
    "yolo11n.pt",
    "yolo11s.pt",
    "yolo11x.pt",
    "yolov8x-worldv2.pt",
    "idd_yolov8.pt",
    "yolov8x-oiv7.pt"
]

# 3. Move files
print("Moving models to /content/models/...")
for file in model_files:
    if os.path.exists(f"/content/{file}"):
        shutil.move(f"/content/{file}", os.path.join(target_dir, file))
        print(f"Moved: {file}")
    else:
        print(f"Warning: {file} not found in /content/")

print("\nAll models have been organized.")

Moving models to /content/models/...
Moved: yolo11n.pt
Moved: yolo11s.pt
Moved: yolo11x.pt
Moved: yolov8x-worldv2.pt
Moved: yolov8x-oiv7.pt

All models have been organized.


# Creating Data.yaml

In [ ]:
import yaml

data_config = {
    'path': '/content/master_data',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'car', 1: 'traffic_light', 2: 'person'}
}

with open('/content/master_data/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

# Training and Refining Model on Master_Data

In [ ]:
from ultralytics import YOLO
import os

# 1. Define model paths
model_files = {
    "yolo26n": "/content/models/yolo11n.pt",
    "yolo26s": "/content/models/yolo11s.pt",
    "yolo26x": "/content/models/yolo11x.pt",
    "lvis_v8": "/content/models/yolov8x-worldv2.pt",
    "idd_v8": "/content/models/idd_yolov8.pt",
    "car_expert": "/content/models/yolov8x-oiv7.pt"

}

# 2. Configuration
save_dir = "/content/trained_models"
training_args = {
    'data': '/content/master_data/data.yaml',
    'epochs': 50,
    'imgsz': 640,
    'batch': 32,
    'optimizer': 'AdamW',
    'verbose': True
}

# 3. Training Loop with Error Handling
for name, model_path in model_files.items():
    if not os.path.exists(model_path):
        print(f"SKIPPING: Model file not found at {model_path}")
        continue

    print(f"--- Training Architecture: {name} ---")

    try:
        # Initialize
        model = YOLO(model_path)

        # Train and save results into the specific 'trained_models' directory
        model.train(**training_args, project=save_dir, name=name)

        # Validate
        metrics = model.val()

        print(f"Successfully finished {name}. mAP@50: {metrics.box.map50:.4f}")

    except Exception as e:
        print(f"Error occurred while training {name}: {e}")

print(f"Training complete. All models saved in {save_dir}")

# Creating Visualization for Accuracy Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Base directory where you saved your projects
base_dir = "detection_results"
model_names = ["yolo26n", "yolo26s", "yolo26x", "lvis_v8", "idd_v8", "car_expert"]

plt.figure(figsize=(15, 6))

for name in model_names:
    results_path = os.path.join(base_dir, name, "results.csv")
    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        # Ultralytics results.csv columns usually have leading spaces
        df.columns = df.columns.str.strip()

        # Plot mAP@50 (Mean Average Precision)
        plt.plot(df['epoch'], df['metrics/mAP50(B)'], label=f"{name} mAP@50")

plt.title("mAP@50 Comparison Across Models")
plt.xlabel("Epochs")
plt.ylabel("mAP@50")
plt.legend()
plt.grid(True)
plt.show()

# Saving Trained Models

In [ ]:
import shutil
from google.colab import files

# Define the folder you want to download and the output zip name
folder_to_zip = '/content/trained_models'
zip_filename = 'trained_models.zip'

# Create the zip file
shutil.make_archive(folder_to_zip, 'zip', folder_to_zip)

# Download the zip file
files.download(f"{folder_to_zip}.zip")

# Testinf Model

In [ ]:
from ultralytics import YOLO

# 1. Load the model
model = YOLO('/content/trained_models/yolo26n/weights/best.pt')

# 2. Run validation on the test folder
# Ensure your data.yaml points to the 'test' directory
metrics = model.val(data='/content/master_data/data.yaml', split='test')

# 3. Access Accuracy Metrics
print(f"mAP@50: {metrics.box.map50}")
print(f"mAP@50-95: {metrics.box.map}")

# Visualization for Training

In [ ]:
from IPython.display import Image, display
import os

# The validation results are saved in the runs/detect/val folder
val_folder = 'runs/detect/val'

# Display the Confusion Matrix
if os.path.exists(os.path.join(val_folder, 'confusion_matrix.png')):
    print("--- Confusion Matrix ---")
    display(Image(filename=os.path.join(val_folder, 'confusion_matrix.png')))

# Display the Precision-Recall Curve
if os.path.exists(os.path.join(val_folder, 'PR_curve.png')):
    print("--- Precision-Recall Curve ---")
    display(Image(filename=os.path.join(val_folder, 'PR_curve.png')))

# Custom Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomDeepNet(nn.Module):
    def __init__(self, num_classes=3):
        super(CustomDeepNet, self).__init__()

        # Layer 1: Initial Convolution
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        # Layer 2: Deeper Conv
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Layer 3: Bottleneck Block
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        # Layer 4: Deep Expansion
        self.layer4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Layer 5: Higher Level Features
        self.layer5 = nn.Sequential(
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU()
        )

        # Layer 6: Final Feature Refinement
        self.layer6 = nn.Sequential(
            nn.Conv2d(1024, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return F.softmax(self.fc(x), dim=1)

# Initialize the 6-layer model
model = CustomDeepNet(num_classes=3)
print(model)

CustomDeepNet(
  (layer1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer4): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1,

# Training on Custome Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import json

class MasterDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # Load label file
        lab_path = os.path.join(self.label_dir, img_name.replace('.jpg', '.txt').replace('.png', '.txt'))

        with open(lab_path, 'r') as f:
            # Read first line and take the first value as class ID
            first_line = f.readline().split()
            class_id = int(first_line[0])

        # STRICT PROTECTION: Force invalid IDs into valid range or skip
        # If your data is 0,1,2, this ensures we don't crash
        if class_id not in [0, 1, 2]:
            # Default to '0' (car) if file is corrupted, or handle accordingly
            class_id = 0

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(class_id, dtype=torch.long)

def run_training(train_dir, val_dir, batch_size=32, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    # Paths based on your structure
    train_dataset = MasterDataset(os.path.join(train_dir, 'images'), os.path.join(train_dir, 'labels'), transform=transform)
    val_dataset = MasterDataset(os.path.join(val_dir, 'images'), os.path.join(val_dir, 'labels'), transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Ensure your CustomDeepNet is defined with num_classes=3
    model = CustomDeepNet(num_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {"train_loss": [], "val_loss": [], "train_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for v_img, v_lbl in val_loader:
                v_img, v_lbl = v_img.to(device), v_lbl.to(device)
                val_loss += criterion(model(v_img), v_lbl).item()

        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        acc = 100 * correct / total

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(acc)

        print(f"Epoch {epoch+1} | Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Acc: {acc:.2f}%")

    torch.save(model.state_dict(), "final_model.pth")
    with open("history.json", "w") as f:
        json.dump(history, f)

# Execute
run_training(train_dir='/content/master_data/train', val_dir='/content/master_data/val', batch_size=32, epochs=50)

# Visualizations

In [ ]:
import matplotlib.pyplot as plt
import json
import torch
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Load the history saved during training
with open("history.json", "r") as f:
    history = json.load(f)

# 2. Plot Training Metrics (Loss and Accuracy)
def plot_performance(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss Graph
    ax1.plot(history['train_loss'], label='Train Loss', marker='o')
    ax1.plot(history['val_loss'], label='Val Loss', marker='o')
    ax1.set_title('Loss Convergence')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy Graph
    ax2.plot(history['train_acc'], label='Train Accuracy', color='green', marker='o')
    ax2.set_title('Training Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.show()

# 3. Generate Confusion Matrix
def plot_confusion_matrix(model, val_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Car', 'Traffic Light', 'Person'],
                yticklabels=['Car', 'Traffic Light', 'Person'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

# --- Execution ---
# Plot Loss and Accuracy
plot_performance(history)

# Re-create the val_loader
val_dataset = MasterDataset(
    os.path.join('/content/master_data/val', 'images'),
    os.path.join('/content/master_data/val', 'labels'),
    transform=transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Plot Confusion Matrix (Requires model and val_loader to be defined)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomDeepNet(num_classes=3).to(device)
model.load_state_dict(torch.load("final_model.pth"))
plot_confusion_matrix(model, val_loader, device)

# Downloading Trained Model and checkpoints

In [ ]:
from google.colab import files

# Download the model
files.download('/content/final_model.pth')

# Download the history
files.download('/content/history.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Testing Model

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os

def test_model(test_dir, model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Re-initialize and load the model
    # Ensure CustomDeepNet is defined in your current environment
    model = CustomDeepNet(num_classes=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. Setup Test Data Loader
    transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    test_dataset = MasterDataset(
        os.path.join(test_dir, 'images'),
        os.path.join(test_dir, 'labels'),
        transform=transform
    )
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds = []
    all_labels = []

    # 3. Inference
    print("Testing on test set...")
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 4. Accuracy & Confusion Matrix
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc * 100:.2f}%")

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Car', 'Traffic Light', 'Person'],
                yticklabels=['Car', 'Traffic Light', 'Person'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

# Run the test
# Replace 'final_model.pth' with your path if different
test_model(test_dir='/content/master_data/test', model_path='final_model.pth')

# Github Repo

https://github.com/KVAlwaysLearning/Car_Color_Detection_Sub

# Streamlit App

https://carcolordetectionsub-app-working.streamlit.app/